Supongamos que nuestro objetivo como analistas de datos es ayudar a una editorial a decidir en qué libros invertir marketing, pero para esto necesitamos definir qué es lo que hace que un libro pueda ser exitoso. Tenemos por un lado libros que tienen un rating muy alto, pero una baja cantidad de reseñas (son poco conocidos) y también libros con miles de reseñas pero un rating más bien mediocre (populares, no necesariamente "buenos"). Entender esta diferencia nos ayudará a hacer mejores decisiones al momento de la inversión.

A tener en cuenta en el análisis:

* El sesgo entre Calidad y Popularidad: la relación entre la calificación promedio y el volumen total de reseñas para identificar qué libros son realmente apreciados por sus lectores versus aquellos que simplemente tienen un alto nivel de alcance.

* Caracterización del Éxito: analizando los distintos perfiles de libros presentes en el catálogo, diferenciando entre éxitos consolidados, productos de consumo masivo, títulos de nicho con alto potencial y libros de bajo desempeño.

* Patrones Directores: investigando si existen atributos en común dentro del catálogo (como la extensión del libro o las características del autor) que influyan en que un libro pase de ser un título poco conocido a un fenómeno de ventas.

* Criterios de Priorización de Inversión: formulación de un marco analítico para que la editorial pueda identificar cuáles son los títulos con mayor margen de crecimiento y retorno de inversión, optimizando así la asignación del presupuesto de marketing.


In [54]:
pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [55]:
pip install matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
pip install seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [57]:
import pandas as pd

In [58]:
df = pd.read_csv('/workspaces/ProyectoFinal-DataAnalysisconPython/data/Goodreads_books_with_genres.csv', on_bad_lines='skip')

In [59]:
df.columns = df.columns.str.strip()

In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11127 entries, 0 to 11126
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Book Id             11127 non-null  int64         
 1   Title               11127 non-null  str           
 2   Author              11127 non-null  str           
 3   average_rating      11127 non-null  float64       
 4   isbn                11127 non-null  str           
 5   isbn13              11127 non-null  int64         
 6   language_code       11127 non-null  str           
 7   num_pages           11127 non-null  int64         
 8   ratings_count       11127 non-null  int64         
 9   text_reviews_count  11127 non-null  int64         
 10  publication_date    11125 non-null  datetime64[us]
 11  publisher           11127 non-null  str           
 12  genres              11030 non-null  str           
 13  publication_year    11125 non-null  float64       
 14  w

In [61]:
df["publication_date"] = pd.to_datetime(df["publication_date"], format="%m/%d/%Y", errors="coerce")
df["publication_year"] = df["publication_date"].dt.year

df.groupby("publication_year").agg(
    cantidad_de_reseñas = ("average_rating", "count"),
    rating_promedio = ("average_rating", "mean")
)

df["publication_year"]

0        2006.0
1        2004.0
2        2003.0
3        2004.0
4        2004.0
          ...  
11122    2004.0
11123    1988.0
11124    1993.0
11125    2007.0
11126    2006.0
Name: publication_year, Length: 11127, dtype: float64

In [63]:
import numpy as np

# 2. Definir constantes globales
C = df['average_rating'].mean()
m = df['ratings_count'].quantile(0.75)  # Exige estar en el 25% superior de populares para no sufrir ponderación

# 3. Vectorizar el cálculo del Promedio Bayesiano
def bayesian_average(df, m, C):
    v = df['ratings_count']
    R = df['average_rating']
    return (v / (v + m)) * R + (m / (v + m)) * C

df['weighted_rating'] = bayesian_average(df, m, C)

# 4. Establecer umbrales de corte para la matriz 2x2
rating_threshold = df['average_rating'].median()
reviews_threshold = df['ratings_count'].median()

# 5. Clasificar en cuadrantes
conditions = [
    (df['average_rating'] >= rating_threshold) & (df['ratings_count'] < reviews_threshold),
    (df['average_rating'] >= rating_threshold) & (df['ratings_count'] >= reviews_threshold),
    (df['average_rating'] < rating_threshold)  & (df['ratings_count'] >= reviews_threshold),
    (df['average_rating'] < rating_threshold)  & (df['ratings_count'] < reviews_threshold)
]

categories = ['Joya Oculta', 'Hitazo', 'Público Masivo', 'Fondo de Catálogo']

df['cuadrante'] = np.select(conditions, categories, default='Sin Clasificar')

# 6. Inspeccionar distribución del catálogo
print(df['cuadrante'].value_counts())

cuadrante
Hitazo               2951
Fondo de Catálogo    2876
Joya Oculta          2687
Público Masivo       2613
Name: count, dtype: int64


In [65]:
df["weighted_rating"]

0        4.568487
1        4.488713
2        4.205575
3        4.558666
4        4.688957
           ...   
11122    3.937459
11123    3.953471
11124    3.937350
11125    3.905122
11126    3.933108
Name: weighted_rating, Length: 11127, dtype: float64